# 🚀 Урок 21 — Старт финального проекта (материалы преподавателя)

Рабочий шаблон: задача → данные → baseline → первичная модель в Pipeline.

> Пример на Palmer Penguins. Ученик подставляет свою задачу по аналогии (табличную или текстовую).

## Шаг 1 · Задача и данные
Формулировка: «предсказываю X по признакам Y, метрика Z».

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
df = sns.load_dataset('penguins').drop(columns=['sex'])   # 👈 пример; sex всегда убираем
df = df.dropna(subset=['species'])                         # target без пропусков
num = ['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']  # 👈 свои числовые
cat = ['island']                                           # 👈 свои категориальные
X = df[num+cat]; y = df['species']                         # 👈 свой target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Строк:', len(X), '| Признаков:', X.shape[1])

## Шаг 2 · Baseline

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
base = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
print(f'Baseline: {accuracy_score(y_te, base.predict(X_te)):.0%}')

## Шаг 3 · Первичная модель в Pipeline (стандарт курса)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
prep = ColumnTransformer([
    ('num', Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]), num),
    ('cat', Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]), cat)])
model = Pipeline([('prep',prep),('rf',RandomForestClassifier(n_estimators=100,random_state=42))])
model.fit(X_tr, y_tr)
print(f'Первичная модель: {accuracy_score(y_te, model.predict(X_te)):.0%}')

## Шаг 4 · Текстовый вариант (для текстовых проектов)
Та же логика, но признак — текст. ⚠️ `Pipeline` (sklearn, большая P) ≠ HF `pipeline`.

In [ ]:
# Демонстрационный текстовый пример (самодостаточный)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
texts = ['отличный фильм','ужасно скучно','мне понравилось','худшее что видел',
         'супер, рекомендую','деньги на ветер','шедевр','полный провал']
labels = ['pos','neg','pos','neg','pos','neg','pos','neg']
Xt_tr,Xt_te,yt_tr,yt_te = train_test_split(texts, labels, test_size=0.25, random_state=42)
text_model = Pipeline([('tfidf', TfidfVectorizer()),
                       ('clf', LogisticRegression(max_iter=1000))])
text_model.fit(Xt_tr, yt_tr)
print('Пример текстовой модели обучен. Прогноз:', text_model.predict(['очень понравилось'])[0])

---
**Итог урока 21.** У каждого ученика есть рабочий старт: задача, данные, baseline, первичная модель. Дальше (урок 22) — улучшение и интерфейс.